# External Validation Set Review

Iterative manual review of the unaltered BridgeData V2 frames held back as the
external validation set, checking that each forms a feasible contrastive pair. Each
session loads the review queue from `manifest.csv`, shows the initial frame (and grasp
frame when present) with both sides of the antonym swap, and writes labels one
scene at a time via `update_manifest_annotations`. Labels live only in the Drive
manifest; this notebook does not hold an annotations dict.

This notebook prepares the closing transfer analysis, not the experiments. Nothing in
the constructed pipeline depends on it, so Notebook 03 can be built and screened
without it. It is placed before the probe for one reason: the labels decide how the
validation scenes are stratified, and fixing them before any prediction exists keeps
them independent of the results they will later be read against.

The queue covers the validation role only. Frames harvested as construction bases
are not reviewed here: their trials are built rather than found, so feasibility
holds by construction and what needs judging instead is whether the composite reads
as a scene, which is the approval screen in Notebook 03.

Each scene also carries a `duplicate_target` label recording whether it holds
two or more instances of the same object, the condition under which an
antonym-swapped instruction names a distinct, real target on both sides. Notebook
02 proposes this label automatically with an open-vocabulary detector. This
notebook is where the borderline `unclear` proposals are confirmed by eye. The
duplicate control below writes `duplicate_target` (and marks the label as
manually sourced) in the same save.

The headline validation stratum requires `feasible_both == 'yes'` and
`duplicate_target == 'yes'`. Other values stay out of that stratum but are kept as
weaker strata. The model and the TFDS stream are never loaded here.

**Prerequisite:** Notebook 02 has harvested frames and a manifest under
`openvla_cache/v2/bridge/`.

## 1. Mount Drive

In [15]:
from google.colab import drive
drive.mount('/content/drive')
import os
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/v2/bridge'
assert os.path.isfile(os.path.join(CACHE_DIR, 'manifest.csv')), (
    f'No manifest at {CACHE_DIR}; run the harvest in Notebook 02 first.'
)
print('cache ->', CACHE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cache -> /content/drive/MyDrive/openvla_cache/v2/bridge


## 2. Import `data.py`

Clones the project code from GitHub into the runtime and imports the loader,
inference, and logging functions from there, so the code always matches the
pushed commit.

In [16]:
import sys, os, importlib, subprocess

REPO_URL = 'https://github.com/LewisTL/ECS8056.git'
BRANCH = 'master'
REPO_DIR = '/content/ECS8056'

def sync_repo():
    """Clone or hard-refresh the repository so it matches origin/BRANCH."""
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url],
                       check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--depth', '1',
                        'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '--quiet',
                        f'origin/{BRANCH}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch',
                        BRANCH, url, REPO_DIR], check=True)
    return subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()


commit = sync_repo()
module_dir = REPO_DIR
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for _m in ('model', 'data', 'export_pairs', 'detect_duplicates'):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()

from data import (
    CATEGORY_OTHER,
    CATEGORY_PLACEMENT,
    CATEGORY_REFERENT,
    DUPLICATE_SOURCE_MANUAL,
    refresh_heuristic_categories,
    review_queue,
    review_summary,
    update_manifest_annotations,
)
print(f'imported data.py from {module_dir} @ {commit}')

imported data.py from /content/ECS8056 @ 41be4c2


## 3. Review progress

Pairable `referent_selection` scenes still marked `unreviewed` are the main
headroom for growing the primary stratum. Re-run this cell after a labeling
session to confirm the counts moved.

The harvest wrote `category` under the rule current at the time. This cell
recomputes that column from the current classifier before printing the
summary, so destination-locative transfer instructions (`put the cup on the
left`) are counted as `placement_relation` rather than remaining in the
referent tally.

In [17]:
refreshed = refresh_heuristic_categories(CACHE_DIR)
print(f"heuristic category refreshed: {refreshed['updated']} of "
      f"{refreshed['total']}")
summary = review_summary(CACHE_DIR)
print(f"cached frames by role: {summary['by_split']}")
print(f"in review scope {summary['reviewed_scope']}: {summary['in_scope']}")
print(f"pairable: {summary['pairable']}  non-pairable: {summary['non_pairable']}")
print(
    f"referent_selection pairable: "
    f"unreviewed={summary['referent_pairable_unreviewed']}  "
    f"yes={summary['referent_pairable_yes']}"
)
print(
    f"duplicate_target (referent pairable): "
    f"yes={summary['referent_pairable_dup_yes']}  "
    f"unclear={summary['referent_pairable_dup_unclear']}  "
    f"unreviewed={summary['referent_pairable_dup_unreviewed']}"
)
print(f"primary_eligible (feasible_both=yes and duplicate_target=yes): "
      f"{summary['primary_eligible']}")
print(f"duplicate_source breakdown: {summary['duplicate_source']}")
print('\nby category x feasible_both:')
for (cat, feas), n in sorted(summary['by_category_feasibility'].items()):
    print(f'  {cat:22} {feas:12} {n}')

[refresh_heuristic_categories] updated 0 of 17035 rows in /content/drive/MyDrive/openvla_cache/v2/bridge/manifest.csv
heuristic category refreshed: 0 of 17035
cached frames by role: {'construction': 16000, 'validation': 1035}
in review scope ('validation',): 1035
pairable: 1035  non-pairable: 0
referent_selection pairable: unreviewed=88  yes=0
duplicate_target (referent pairable): yes=0  unclear=40  unreviewed=0
primary_eligible (feasible_both=yes and duplicate_target=yes): 0
duplicate_source breakdown: {'manual': 12, 'auto': 88}

by category x feasible_both:
  placement_relation     no           20
  placement_relation     unreviewed   915
  referent_selection     no           12
  referent_selection     unreviewed   88


## 4. Interactive reviewer

Judge whether the implied placement or referent choice is physically possible
on **both** sides of the antonym swap, and confirm whether the scene holds two
or more instances of the same object (`duplicate_target`). Default queue:
unreviewed pairable `referent_selection` scenes the detector marked
`duplicate_target='unclear'`. That is the borderline band, and it is larger
than the auto-`yes` set.

Review in three passes by changing `DUPLICATE_STATUS`:
- `'yes'`: detector is confident there are two of the named object (a handful
  of frames after placement is excluded).
- `'unclear'`: borderline second-instance score; this is the main remaining
  pool.
- `'no'` or `None`: auto-singles and the full referent set. Low yield; only
  useful as an audit of the detector.

Change `CATEGORIES` to `None` to include placement scenes, or set `STATUS` to
revisit `yes`, `no`, or `unclear`.

Each save writes a single row to the Drive manifest and advances. The duplicate
control is saved together with the feasibility choice and marks the label as
manually sourced. Skip leaves the row unchanged.

In [18]:
import ipywidgets as widgets
from IPython.display import display

# --- Session filters (edit and re-run this cell to change the queue) ---------
STATUS = 'unreviewed'       # feasible_both: 'yes'/'no'/'unclear'/None for all
DUPLICATE_STATUS = 'unclear'  # 'yes' then 'unclear'; None includes auto-no
CATEGORIES = [CATEGORY_REFERENT]  # None to include placement scenes
ONLY_PAIRABLE = True
BATCH_SIZE = 300             # max scenes loaded for this session

queue = review_queue(
    CACHE_DIR,
    status=STATUS,
    duplicate_status=DUPLICATE_STATUS,
    categories=CATEGORIES,
    only_pairable=ONLY_PAIRABLE,
    limit=BATCH_SIZE,
)
print(f'queue length: {len(queue)} (status={STATUS!r}, '
      f'duplicate_status={DUPLICATE_STATUS!r}, limit={BATCH_SIZE})')
if not queue:
    print('Nothing to review under the current filters.')

state = {'idx': 0, 'done': 0}

status_label = widgets.HTML()
meta_label = widgets.HTML()
# Frames are `widgets.Image` children of this panel, not matplotlib figures
# written into an `Output`. Colab does not render that combination, so that
# pattern leaves the reviewer with the controls and no image. Replacing the
# panel's children is the same pattern as the approval screen in Notebook 03.
img_panel = widgets.HBox([])
note_box = widgets.Text(
    description='Note',
    placeholder='optional feasibility note',
    layout=widgets.Layout(width='90%'),
)
cat_dd = widgets.Dropdown(
    options=[
        ('(keep heuristic)', ''),
        (CATEGORY_REFERENT, CATEGORY_REFERENT),
        (CATEGORY_PLACEMENT, CATEGORY_PLACEMENT),
        (CATEGORY_OTHER, CATEGORY_OTHER),
    ],
    description='Category',
    layout=widgets.Layout(width='50%'),
)
dup_dd = widgets.ToggleButtons(
    options=[
        ('duplicate: unreviewed', 'unreviewed'),
        ('duplicate: yes', 'yes'),
        ('duplicate: no', 'no'),
        ('duplicate: unclear', 'unclear'),
    ],
    description='Duplicate',
    tooltips=[
        'not yet judged',
        'two or more identical objects present',
        'single target of this type',
        'cannot tell from the frame',
    ],
)
btn_yes = widgets.Button(description='Yes', button_style='success')
btn_no = widgets.Button(description='No', button_style='danger')
btn_unclear = widgets.Button(description='Unclear', button_style='warning')
btn_skip = widgets.Button(description='Skip')
buttons = widgets.HBox([btn_yes, btn_no, btn_unclear, btn_skip])


def _frame_panel(rel: str, title: str):
    """Return a labelled PNG widget, or a missing-file note."""
    caption = widgets.HTML(f'<b>{title}</b>')
    path = os.path.join(CACHE_DIR, rel) if rel else ''
    if not rel or not os.path.isfile(path):
        return widgets.VBox([
            caption,
            widgets.HTML(f'missing: <code>{rel or "(empty path)"}</code>'),
        ])
    with open(path, 'rb') as f:
        data = f.read()
    return widgets.VBox([
        caption,
        widgets.Image(value=data, format='png', width=480),
    ])


def _show_current():
    if state['idx'] >= len(queue):
        status_label.value = (
            f"<b>Session complete.</b> Labeled {state['done']} scene(s) this run."
        )
        meta_label.value = ''
        note_box.value = ''
        img_panel.children = ()
        return
    item = queue[state['idx']]
    remaining = len(queue) - state['idx']
    status_label.value = (
        f"Scene {state['idx'] + 1} / {len(queue)} "
        f"(remaining in batch: {remaining}; labeled this run: {state['done']})"
    )
    dup_src = item.get('duplicate_source', '') or 'none'
    dup_score = item.get('duplicate_score', '') or ''
    meta_label.value = (
        f"<b>ep {item['episode_index']}</b> "
        f"[{item['category']}] term=<code>{item['spatial_term']}</code><br>"
        f"A: {item['instruction']}<br>"
        f"B: {item['instr_b']}<br>"
        f"current feasible_both=<code>{item['feasible_both']}</code>  "
        f"duplicate_target=<code>{item['duplicate_target']}</code> "
        f"(source={dup_src}, score={dup_score})"
    )
    note_box.value = item.get('feasibility_note', '') or ''
    cat_dd.value = item.get('category_manual', '') or ''
    dup_dd.value = item.get('duplicate_target', '') or 'unreviewed'
    paths = [item['image_path']]
    titles = ['initial']
    if item.get('grasp_image_path'):
        paths.append(item['grasp_image_path'])
        titles.append('grasp')
    img_panel.children = tuple(
        _frame_panel(rel, title) for rel, title in zip(paths, titles)
    )


def _save(feasible: str):
    if state['idx'] >= len(queue):
        return
    item = queue[state['idx']]
    ep = int(item['episode_index'])
    note = {'feasible_both': feasible}
    text = note_box.value.strip()
    if text:
        note['feasibility_note'] = text
    if cat_dd.value:
        note['category_manual'] = cat_dd.value
    # Record the duplicate-target judgement whenever it moves off unreviewed,
    # and mark the label as manually sourced so it is distinguishable from an
    # automated proposal.
    if dup_dd.value and dup_dd.value != 'unreviewed':
        note['duplicate_target'] = dup_dd.value
        note['duplicate_source'] = DUPLICATE_SOURCE_MANUAL
        item['duplicate_target'] = dup_dd.value
        item['duplicate_source'] = DUPLICATE_SOURCE_MANUAL
    update_manifest_annotations(CACHE_DIR, {ep: note})
    item['feasible_both'] = feasible
    state['done'] += 1
    state['idx'] += 1
    _show_current()


def _skip(_):
    if state['idx'] >= len(queue):
        return
    state['idx'] += 1
    _show_current()


btn_yes.on_click(lambda _: _save('yes'))
btn_no.on_click(lambda _: _save('no'))
btn_unclear.on_click(lambda _: _save('unclear'))
btn_skip.on_click(_skip)

ui = widgets.VBox([status_label, meta_label, img_panel, note_box, cat_dd, dup_dd,
                   buttons])
display(ui)
_show_current()

queue length: 40 (status='unreviewed', duplicate_status='unclear', limit=300)


## 5. Progress after the session

Re-print the summary so the session's writes are visible without reloading
the notebook.

In [20]:
summary = review_summary(CACHE_DIR)
print(f"cached frames by role: {summary['by_split']}")
print(f"in review scope {summary['reviewed_scope']}: {summary['in_scope']}")
print(f"pairable: {summary['pairable']}  non-pairable: {summary['non_pairable']}")
print(
    f"referent_selection pairable: "
    f"unreviewed={summary['referent_pairable_unreviewed']}  "
    f"yes={summary['referent_pairable_yes']}"
)
print(
    f"duplicate_target (referent pairable): "
    f"yes={summary['referent_pairable_dup_yes']}  "
    f"unclear={summary['referent_pairable_dup_unclear']}  "
    f"unreviewed={summary['referent_pairable_dup_unreviewed']}"
)
print(f"primary_eligible (feasible_both=yes and duplicate_target=yes): "
      f"{summary['primary_eligible']}")
print(f"duplicate_source breakdown: {summary['duplicate_source']}")
print('\nby category x feasible_both:')
for (cat, feas), n in sorted(summary['by_category_feasibility'].items()):
    print(f'  {cat:22} {feas:12} {n}')

cached frames by role: {'construction': 16000, 'validation': 1035}
in review scope ('validation',): 1035
pairable: 1035  non-pairable: 0
referent_selection pairable: unreviewed=48  yes=0
duplicate_target (referent pairable): yes=0  unclear=0  unreviewed=0
primary_eligible (feasible_both=yes and duplicate_target=yes): 0
duplicate_source breakdown: {'manual': 40, 'auto': 48}

by category x feasible_both:
  placement_relation     no           32
  placement_relation     unreviewed   915
  referent_selection     no           40
  referent_selection     unreviewed   48
